# NX-421 Mini-project - Part 1: structural and functional preprocessing
Lab code (Labs 2, 3, 4) inlined and adapted to our data. FSL tools take file names with or without `.nii.gz`; nibabel needs the full name.

## 0. Setup

In [ ]:
# ---- Make utils.py importable, load FSL in Neurodesk, import only what we use ----
import os, sys, subprocess
import os.path as op

notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)                     # utils.py lives one folder up
from utils import loadFSL, FSLeyesServer

import module as lmod                           # Neurodesk module system
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')                 # puts bet, fast, flirt, fnirt, ... on the PATH
loadFSL()                                       # sets the FSL environment variables

from fsl.wrappers import fast, flirt, fslroi, mcflirt   # Python wrappers around FSL tools
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
%gui wx

In [ ]:
# ---- Start FSLeyes (viewer used for all visual QC) ----
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
# ---- Paths ----
data_dir   = "/home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final"  # raw data, read only
output_dir = op.join(parent_dir, "outputs")     # derivatives on local disk, NOT on tsclient
anat_dir   = op.join(output_dir, "anat")
func_dir   = op.join(output_dir, "func")
os.makedirs(anat_dir, exist_ok=True)            # also creates output_dir
os.makedirs(func_dir, exist_ok=True)

t1_path   = op.join(data_dir, "T1", "T1w.nii.gz")
fmri_path = op.join(data_dir, "fMRI", "tfMRI_LANGUAGE_LR", "tfMRI_LANGUAGE_LR.nii.gz")

std_dir       = op.expandvars("$FSLDIR/data/standard")
mni_brain_1mm = op.join(std_dir, "MNI152_T1_1mm_brain")   # FLIRT reference (skull-stripped)
mni_2mm       = op.join(std_dir, "MNI152_T1_2mm")         # FNIRT reference (whole head)

print("T1 found:", op.isfile(t1_path), "| fMRI found:", op.isfile(fmri_path))

## 1. Structural preprocessing
### 1a. Skull stripping (BET)

In [ ]:
# BET fits a surface to the brain edge and zeroes everything outside it
t1_brain = op.join(anat_dir, "T1w_brain")                     # output prefix; -m also writes T1w_brain_mask
subprocess.run(["bet", t1_path, t1_brain, "-m"], check=True)  # add "-R" (robust centre) if QC shows a poor fit

# QC / report figure: skull-stripped brain over the original T1
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path)
fsleyesDisplay.load(t1_brain)

### 1b. Tissue segmentation (FAST)

In [ ]:
# FAST models each voxel as a mix of 3 tissues (on the skull-stripped brain)
fast_prefix = op.join(anat_dir, "T1w_fast")
fast(imgs=[t1_brain], out=fast_prefix, n_classes=3)   # -> _pve_0 (CSF), _pve_1 (GM), _pve_2 (WM)

# QC / report figure: tissue maps over the brain
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_brain)
fsleyesDisplay.load(fast_prefix + "_pve_0")
fsleyesDisplay.load(fast_prefix + "_pve_1")
fsleyesDisplay.load(fast_prefix + "_pve_2")
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'    # CSF
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'  # GM
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'   # WM

### 1c. Registration to MNI (FLIRT, then FNIRT)

In [ ]:
# Report figure "before": native T1 over the template (FSLeyes places both by world coordinates)
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(mni_brain_1mm)
fsleyesDisplay.load(t1_brain)

In [ ]:
# Linear (affine, 12 dof by default): brain -> brain template, two within-modality costs
t1_mni_lin = op.join(anat_dir, "T1w_mni_lin")
for c in ['leastsq', 'normcorr']:
    flirt(t1_brain, mni_brain_1mm, out=t1_mni_lin + '_' + c, omat=t1_mni_lin + '_' + c + '.mat', cost=c)

# QC: both results on the template, then choose one below
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(mni_brain_1mm)
for c in ['leastsq', 'normcorr']:
    fsleyesDisplay.load(t1_mni_lin + '_' + c)

In [ ]:
# Non-linear: start from the chosen affine; whole-head T1 -> whole-head 2 mm template
best_cost     = 'normcorr'                                # set after the visual QC above
t1_mni_nonlin = op.join(anat_dir, "T1w_mni_nonlin")       # warped T1 (2 mm MNI grid)
t1_mni_warp   = op.join(anat_dir, "T1w_to_MNI_warpcoef")  # the warp itself, reused in 2c
subprocess.run(["fnirt", "--in=" + t1_path, "--aff=" + t1_mni_lin + '_' + best_cost + '.mat',
                "--ref=" + mni_2mm, "--config=T1_2_MNI152_2mm",
                "--iout=" + t1_mni_nonlin, "--cout=" + t1_mni_warp], check=True)

# Report figure "after": linear vs non-linear result on the template
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(mni_2mm)
fsleyesDisplay.load(t1_mni_lin + '_' + best_cost)
fsleyesDisplay.load(t1_mni_nonlin)

## 2. Functional preprocessing
### 2a. Volume removal and motion correction

In [ ]:
# Header only (data is loaded lazily): shape, voxel size and TR
img = nib.load(fmri_path)
print("Shape:", img.shape, "| voxel size (mm) and TR (s):", img.header.get_zooms())

# Mean intensity per volume: non-steady-state volumes stand out at the start
plt.plot(img.get_fdata(dtype=np.float32).mean(axis=(0, 1, 2)))
plt.xlabel('Time (volume)'); plt.ylabel('Mean voxel intensity'); plt.show()

In [ ]:
# Crop in time: keep everything from start_vol on (0-based)
start_vol = 1                                             # choose from the plot above
settled = op.join(func_dir, "tfMRI_LANGUAGE_LR_settled")
fslroi(fmri_path, settled, str(start_vol), str(img.shape[-1] - start_vol))
# NB for the GLM: every event onset must be shifted by start_vol * TR

# Rigid-body (6 dof) realignment of every volume to the middle volume
moco = op.join(func_dir, "tfMRI_LANGUAGE_LR_moco")
mcflirt(infile=settled, o=moco, plots=True, report=True, dof=6, mats=True)

In [ ]:
# Motion parameters (.par): 3 rotations (rad) + 3 translations (mm), one row per volume
mot_params = pd.read_csv(moco + '.par', sep='  ', header=None, engine='python',
                         names=['Rotation x', 'Rotation y', 'Rotation z', 'Translation x', 'Translation y', 'Translation z'])

# Power FD: frame-to-frame change, rotations converted to mm on a 50 mm sphere
framewise_diff = mot_params.diff().iloc[1:]
converted_rots = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']] * 50
trans_params   = framewise_diff[['Translation x', 'Translation y', 'Translation z']]
fd = (converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)).to_numpy()

# Data-driven outlier threshold: Q3 + 1.5 * IQR
threshold = np.quantile(fd, 0.75) + 1.5*(np.quantile(fd, 0.75) - np.quantile(fd, 0.25))
outliers  = np.where(fd > threshold)[0] + 1      # +1: fd[0] is the move from volume 0 to volume 1

# Report figure: FD over time with the threshold
plt.plot(range(1, fd.size + 1), fd)
plt.hlines(threshold, 0, fd.size, colors='black', linestyles='dashed', label='FD threshold')
plt.xlabel('Volume'); plt.ylabel('FD displacement (mm)'); plt.legend(); plt.show()
print(f"Mean FD {fd.mean():.3f} mm | max {fd.max():.3f} mm | threshold {threshold:.3f} mm | above: {outliers.tolist()}")
np.savetxt(op.join(func_dir, "fd_outliers.txt"), outliers, fmt="%d")   # kept for censoring in the GLM

### 2b. Gaussian smoothing

In [ ]:
# 6 mm FWHM (about 3x the 2 mm voxel); fslmaths takes sigma = FWHM / 2.3548
smoothed = moco + "_smoothed-6mm"
subprocess.run(['fslmaths', moco, '-s', str(6/2.3548), smoothed], check=True)

# Report figure: the same volume before and after smoothing
vol = 100
fslroi(moco, moco + "_vol100", str(vol), str(1))
fslroi(smoothed, smoothed + "_vol100", str(vol), str(1))
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(moco + "_vol100")
fsleyesDisplay.load(smoothed + "_vol100")

### 2c. Functional-anatomical alignment

In [ ]:
# Middle volume of the motion-corrected run (= MCFLIRT's reference volume)
epi_vol = moco + "_vol_middle"
fslroi(moco, epi_vol, str((img.shape[-1] - start_vol) // 2), str(1))

# Is the EPI already on the MNI 2 mm grid? (same shape and same affine as the template)
epi, mni = nib.load(epi_vol + ".nii.gz"), nib.load(mni_2mm + ".nii.gz")
already_mni = epi.shape[:3] == mni.shape and np.allclose(epi.affine, mni.affine)
print("EPI already in MNI space:", already_mni)
print("EPI affine:\n", epi.affine, "\nMNI 2 mm affine:\n", mni.affine)

In [ ]:
epi_check = epi_vol
if not already_mni:
    # EPI -> T1 with boundary-based registration (whole-head T1, brain T1, WM map from FAST)
    epi2t1 = op.join(func_dir, "epi_to_T1")
    subprocess.run(['epi_reg', '--epi={}'.format(epi_vol), '--t1={}'.format(t1_path),
                    '--t1brain={}'.format(t1_brain), '--out={}'.format(epi2t1),
                    '--wmseg={}'.format(fast_prefix + '_pve_2.nii.gz')], check=True)
    # EPI -> MNI with one interpolation: EPI->T1 matrix first, then the T1->MNI FNIRT warp
    epi_check = op.join(func_dir, "epi_vol_middle_MNI")
    subprocess.run(['applywarp', '-i', epi_vol, '-r', mni_2mm, '-o', epi_check,
                    '-w', t1_mni_warp, '--premat={}'.format(epi2t1 + '.mat')], check=True)

# Report figure: EPI over the T1 normalised to MNI
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_mni_nonlin)
fsleyesDisplay.load(epi_check)